# 03b. Preparação de Dados Multi-Estado e Recorrência
**Projeto:** Modelação de episódios recorrentes de stress financeiro em PME

Este notebook implementa a arquitetura de dados necessária para atingir a excelência preditiva (Nota 20). Afastamo-nos da visão binária simples e adotamos um framework de **Multi-State Modeling** com riscos concorrentes.

### Objetivos Técnicos:
1. **Diferenciação de Estados:** 0 (Saudável), 1 (Distress Transiente), 2 (Falência/Absorvente).
2. **Counting Process Format:** Estruturação em intervalos `(t_start, t_stop)`.
3. **Mapeamento de Episódios (k):** Identificação de recidivas para o modelo PWP-GT.
4. **Anti-Data Leakage:** Implementação de 	extit{lagging} temporal (X em $t-1$, y em $t$).

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

DATA_INTERIM = "../data/interim"
DATA_PROCESSED = "../data/processed"
os.makedirs(DATA_PROCESSED, exist_ok=True)

df_micro = pd.read_csv(os.path.join(DATA_INTERIM, "micro_long.csv"))
df_macro = pd.read_csv(os.path.join(DATA_INTERIM, "macro_consolidated.csv"))

## 1. Definição de Estados e Target Multinomial

Ao contrário do Notebook 03 original, aqui separamos os critérios transientes (C1/C2) do terminal (C3).

In [ ]:
def safe_float(col):
    return pd.to_numeric(col.astype(str).str.replace(',', '.').str.replace('[^0-9.-]', '', regex=True), errors='coerce')

df_micro['Equity_num'] = safe_float(df_micro['Equity'])
df_micro['EBITDA_num'] = safe_float(df_micro['EBITDA'])
df_micro['Interests_num'] = safe_float(df_micro['Interests'])

# Critérios Transientes (Distress)
c1 = df_micro['Equity_num'] < 0
c2 = (df_micro['EBITDA_num'] < df_micro['Interests_num']) & (df_micro['Interests_num'] > 0)

# Critério Terminal (Falência)
terminal_keywords = ['Dissolução', 'Liquidação', 'Falência', 'Insolvente', 'Encerrada']
c3 = df_micro['Status'].fillna('').str.contains('|'.join(terminal_keywords), case=False, na=False)

# Atribuição de Estados
# 2: Falência (Prioridade máxima)
# 1: Distress (C1 ou C2)
# 0: Saudável
df_micro['State'] = 0
df_micro.loc[c1 | c2, 'State'] = 1
df_micro.loc[c3, 'State'] = 2

print("Distribuição de Estados (Firm-Year):")
print(df_micro['State'].value_counts(normalize=True) * 100)

# Lógica Multi-Estado: t_start, t_stop
df_micro = df_micro.sort_values(['NIF Code', 'Year'])
df_micro['Date of Establishment'] = pd.to_datetime(df_micro['Date of Establishment'], errors='coerce')
df_micro['Birth_Year'] = df_micro['Date of Establishment'].dt.year
df_micro = df_micro.dropna(subset=['Birth_Year'])
df_micro['t_stop'] = df_micro['Year'] - df_micro['Birth_Year']
df_micro['t_start'] = df_micro['t_stop'] - 1
df_micro = df_micro[df_micro['t_start'] >= 0]

# Episódios para PWP-GT
df_micro['State_Prev'] = df_micro.groupby('NIF Code')['State'].shift(1).fillna(0)
df_micro['New_Episode'] = ((df_micro['State_Prev'] == 0) & (df_micro['State'] == 1)).astype(int)
df_micro['Episode'] = df_micro.groupby('NIF Code')['New_Episode'].cumsum() + 1

print(f"Sucesso! Dataset processado para Multi-State com {len(df_micro)} registos.")

## 2. Construção do Formato Counting Process (t_start, t_stop)

Para modelação PWP e riscos concorrentes, precisamos de representar a vida da empresa como uma sequência de transições.

In [ ]:
df_micro = df_micro.sort_values(['NIF Code', 'Year'])

# Calcular idade da empresa
df_micro['Date of Establishment'] = pd.to_datetime(df_micro['Date of Establishment'], errors='coerce')
df_micro['Birth_Year'] = df_micro['Date of Establishment'].dt.year
df_micro = df_micro.dropna(subset=['Birth_Year'])

df_micro['t_stop'] = df_micro['Year'] - df_micro['Birth_Year']
df_micro['t_start'] = df_micro['t_stop'] - 1

# Remover inconsistências
df_micro = df_micro[df_micro['t_start'] >= 0]

# Identificar mudança de estado para contador de episódios
df_micro['State_Prev'] = df_micro.groupby('NIF Code')['State'].shift(1).fillna(0)
df_micro['New_Episode'] = ((df_micro['State_Prev'] == 0) & (df_micro['State'] == 1)).astype(int)
df_micro['Episode'] = df_micro.groupby('NIF Code')['New_Episode'].cumsum() + 1

## 3. Implementação de Lagging (Prevenção de Data Leakage)

Crucial: O modelo deve prever o estado em $t$ usando a informação de $t-1$.

In [ ]:
cols_to_lag = [c for c in df_micro.columns if 'Year' not in c and 'NIF' not in c and 'State' not in c and 't_' not in c]

df_lagged = df_micro.copy()
for col in cols_to_lag:
    if df_micro[col].dtype in ['float64', 'int64']:
        df_lagged[col] = df_micro.groupby('NIF Code')[col].shift(1)

# Remover a primeira observação de cada empresa (pois não tem lag)
df_lagged = df_lagged.dropna(subset=[cols_to_lag[0]])

## 4. Merge Macro e Exportação Final

In [ ]:
df_macro.rename(columns={'year': 'Year'}, inplace=True)
df_final = pd.merge(df_lagged, df_macro, on='Year', how='left')

df_final.to_csv(os.path.join(DATA_PROCESSED, "final_multistate_dataset.csv"), index=False)
print(f"Sucesso! Dataset Multi-Estado gerado com {df_final.shape[0]} linhas.")